# Introducción a pandas

En este notebook introducimos los conceptos básicos necesarios para trabajar con datos tabulares en Python.

Trabajaremos con `pandas`, una de las librerías más utilizadas para análisis de datos.

---

## Objetivos

- Leer datos desde ficheros
- Entender la estructura de un DataFrame
- Acceder a filas y columnas
- Filtrar datos
- Agregar información
- Combinar datasets (merge)
- Reestructurar datos (pivot)

## Leer datos

In [28]:
import json
import pandas as pd

df = pd.read_csv("data/covid19/casos_hosp_uci_def_sexo_edad_provres.csv")


# cargamos el diccionario de códigos de provincias
with open("data/covid19/cod_provinces.json") as f:
    cod_pro_dict = json.load(f)

# Convertimos los Ids de provincias a códigos del INE (necesario para el merge con población)
df['provincia_iso'] = df['provincia_iso'].map(cod_pro_dict)

df.head()

,provincia_iso,sexo,grupo_edad,fecha,num_casos,num_hosp,num_uci,num_def
0,03,H,0-9,2020-01-01,0,0,0,0
1,03,H,10-19,2020-01-01,0,0,0,0
2,03,H,20-29,2020-01-01,0,0,0,0
3,03,H,30-39,2020-01-01,0,0,0,0
4,03,H,40-49,2020-01-01,0,0,0,0


In [6]:
print("Dataframe shape:", df.shape)
print()
for i in df.columns:
    print(f"- {i}")
print()
df.head()

Dataframe shape: (1299030, 8)

- provincia_iso
- sexo
- grupo_edad
- fecha
- num_casos
- num_hosp
- num_uci
- num_def



,provincia_iso,sexo,grupo_edad,fecha,num_casos,num_hosp,num_uci,num_def
0,A,H,0-9,2020-01-01,0,0,0,0
1,A,H,10-19,2020-01-01,0,0,0,0
2,A,H,20-29,2020-01-01,0,0,0,0
3,A,H,30-39,2020-01-01,0,0,0,0
4,A,H,40-49,2020-01-01,0,0,0,0


## Selección de columnas

In [ ]:
# Seleccionamos una columna
df["num_casos"].head()

,provincia_iso,num_casos
0,A,0
1,A,0
2,A,0
3,A,0
4,A,0


In [ ]:
# Seleccionamos varias columnas
df[["provincia_iso", "num_casos"]].head()

## Filtrado de datos

In [10]:
df[df["provincia_iso"] == "A"].head()

,provincia_iso,sexo,grupo_edad,fecha,num_casos,num_hosp,num_uci,num_def
0,A,H,0-9,2020-01-01,0,0,0,0
1,A,H,10-19,2020-01-01,0,0,0,0
2,A,H,20-29,2020-01-01,0,0,0,0
3,A,H,30-39,2020-01-01,0,0,0,0
4,A,H,40-49,2020-01-01,0,0,0,0


In [11]:
df[df["num_casos"] > 100].head()

,provincia_iso,sexo,grupo_edad,fecha,num_casos,num_hosp,num_uci,num_def
116887,M,H,70-79,2020-03-14,106,103,18,11
118477,M,H,70-79,2020-03-15,105,117,13,10
118478,M,H,80+,2020-03-15,106,105,2,31
120066,M,H,60-69,2020-03-16,106,95,14,5
120067,M,H,70-79,2020-03-16,171,199,22,19


## Indexado
Existe distintas formas de indexado

In [13]:
# Un alemento específico
df.iloc[0]

# Un subconjunto de filas
df.iloc[0:5]

,provincia_iso,sexo,grupo_edad,fecha,num_casos,num_hosp,num_uci,num_def
0,A,H,0-9,2020-01-01,0,0,0,0
1,A,H,10-19,2020-01-01,0,0,0,0
2,A,H,20-29,2020-01-01,0,0,0,0
3,A,H,30-39,2020-01-01,0,0,0,0
4,A,H,40-49,2020-01-01,0,0,0,0


## Agrupación de datos

In [14]:
df.groupby("provincia_iso")["num_casos"].sum().head()

provincia_iso
A      467681
AB      88633
AL     136438
AV      39809
B     1756208
Name: num_casos, dtype: int64

## Agrupación por varias columnas

In [15]:
df.groupby(["fecha", "provincia_iso"])["num_casos"].sum().head()

fecha       provincia_iso
2020-01-01  A                0
            AB               0
            AL               0
            AV               0
            B                0
Name: num_casos, dtype: int64

## Reset index

In [17]:
df_agg = df.groupby(["fecha", "provincia_iso"])["num_casos"].sum().reset_index()
df_agg.head()

,fecha,provincia_iso,num_casos
0,2020-01-01,A,0
1,2020-01-01,AB,0
2,2020-01-01,AL,0
3,2020-01-01,AV,0
4,2020-01-01,B,0


## Combinar datasets (merge)

In [29]:
pop = pd.read_parquet("data/covid19/provincias.parquet")
pop = pop[["name", "total"]].rename(columns={"total": "poblacion"})

covid_total = df.groupby("provincia_iso")["num_casos"].sum().reset_index()

merged = covid_total.merge(pop, left_on="provincia_iso", right_index=True)
merged.head()
covid_total

,provincia_iso,num_casos
0,01,94121
1,02,88633
2,03,467681
3,04,136438
4,05,39809
5,06,164385
6,07,275395
7,08,1756208
8,09,106301
9,10,87156


## Reestructurar datos (pivot)

In [31]:
pivot = df_agg.pivot(
    index="fecha",
    columns="provincia_iso",
    values="num_casos"
)

pivot.head()

provincia_iso,A,AB,AL,AV,B,BA,BI,BU,C,CA,...,SS,T,TE,TF,TO,V,VA,VI,Z,ZA
fecha,,,,,,,,,,,,,,,,,,,,,
2020-01-01,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2020-01-02,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2020-01-03,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2020-01-04,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2020-01-05,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Para una introducción más profunda revisar los siguientes recursos:
- https://pandas.pydata.org/docs/user_guide/10min.html
- https://github.com/mrdbourke/zero-to-mastery-ml/blob/master/section-2-data-science-and-ml-tools/introduction-to-pandas.ipynb
- https://colab.research.google.com/github/waterhackweek/learning-resources/blob/master/notebooks/pandas-intro.ipynb